## Michal Issacs 211371331 
## Levona Tal 324863919 ##

In [2]:
pip install requests beautifulsoup4 python-dotenv pandas

     -------------------------------------- 64.7/64.7 KB 698.7 kB/s eta 0:00:00
     -------------------------------------- 107.7/107.7 KB 3.0 MB/s eta 0:00:00
     ---------------------------------------- 11.4/11.4 MB 4.7 MB/s eta 0:00:00
     ---------------------------------------- 68.6/68.6 KB 3.6 MB/s eta 0:00:00
     -------------------------------------- 135.7/135.7 KB 4.0 MB/s eta 0:00:00
     -------------------------------------- 131.6/131.6 KB 7.6 MB/s eta 0:00:00
     -------------------------------------- 154.7/154.7 KB 4.7 MB/s eta 0:00:00
     -------------------------------------- 510.1/510.1 KB 6.4 MB/s eta 0:00:00
     -------------------------------------- 349.3/349.3 KB 7.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\micha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip' command.


In [3]:
import requests
from bs4 import BeautifulSoup
import json
import re
import urllib.parse
from dotenv import load_dotenv
import os
import pandas as pd

## Import the basics table ##

In [6]:
url_basics = "https://datasets.imdbws.com/title.basics.tsv.gz"
df = pd.read_csv(
    url_basics,
    sep="\t",
    compression="gzip",
    nrows=5
)

print(df)

      tconst titleType            primaryTitle           originalTitle  \
0  tt0000001     short              Carmencita              Carmencita   
1  tt0000002     short  Le clown et ses chiens  Le clown et ses chiens   
2  tt0000003     short            Poor Pierrot          Pauvre Pierrot   
3  tt0000004     short             Un bon bock             Un bon bock   
4  tt0000005     short        Blacksmith Scene        Blacksmith Scene   

   isAdult  startYear endYear  runtimeMinutes                    genres  
0        0       1894      \N               1         Documentary,Short  
1        0       1892      \N               5           Animation,Short  
2        0       1892      \N               5  Animation,Comedy,Romance  
3        0       1892      \N              12           Animation,Short  
4        0       1893      \N               1                     Short  


In [7]:
url_basics = "https://datasets.imdbws.com/title.basics.tsv.gz"

chunks = pd.read_csv(
    url_basics,
    sep="\t",
    compression="gzip",
    na_values="\\N",
    usecols=["tconst", "titleType", "primaryTitle", "startYear", "genres","runtimeMinutes"],
    chunksize=1000
)

movies_list = []

for chunk in chunks:
    movies = chunk[
        (chunk["titleType"] == "movie") &
        (chunk["startYear"].notna())
    ]
    
    movies_list.append(movies)

df_basics = pd.concat(movies_list, ignore_index=True)

df_basics.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
0,tt0000009,movie,Miss Jerry,1894.0,45.0,Romance
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897.0,100.0,"Documentary,News,Sport"
2,tt0000502,movie,Bohemios,1905.0,100.0,NaN
3,tt0000574,movie,The Story of the Kelly Gang,1906.0,70.0,"Action,Adventure,Biography"
4,tt0000591,movie,The Prodigal Son,1907.0,90.0,Drama


## Filtering the table ##

In [8]:
df_basics_filtered  = df_basics[df_basics['titleType'] == 'movie']

In [9]:
df_basics_filtered = df_basics_filtered[df_basics_filtered['primaryTitle'].str.startswith(('I','i', 'J', 'j'), na=False)]

In [10]:
df_basics_filtered.loc[:, 'runtimeMinutes'] = pd.to_numeric(df_basics_filtered['runtimeMinutes'], errors='coerce')
df_basics_filtered= df_basics_filtered[df_basics_filtered['runtimeMinutes'].between(60, 300, inclusive='both')]

In [11]:
df_basics_filtered.loc[:, 'startYear'] = pd.to_numeric(df_basics_filtered['startYear'], errors='coerce')
df_basics_filtered = df_basics_filtered[df_basics_filtered['startYear']<=2024]

In [12]:
df_basics_filtered.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
258,tt0003014,movie,Ingeborg Holm,1913.0,96.0,Drama
510,tt0004022,movie,Julius Caesar,1914.0,112.0,"Drama,History"
545,tt0004150,movie,In the Land of the Head Hunters,1914.0,65.0,"Drama,History"
559,tt0004181,movie,Judith of Bethulia,1914.0,61.0,Drama
1133,tt0005541,movie,In the Palace of the King,1915.0,60.0,"Drama,History"


In [13]:
df_basics_filtered.count()

tconst            18055
titleType         18055
primaryTitle      18055
startYear         18055
runtimeMinutes    18055
genres            16723
dtype: int64

In [14]:
df_basics_filtered[df_basics_filtered["startYear"] == 2024].head() #Test on one film

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
228426,tt10206898,movie,"Il bambino, i due furfanti e la strega",2024.0,95.0,"Drama,Horror"
229115,tt10243672,movie,Joy,2024.0,115.0,"Biography,Drama,History"
233983,tt10482654,movie,I'm 'George Lucas': A Connor Ratliff Story,2024.0,90.0,Documentary
234426,tt10504264,movie,Jenni,2024.0,113.0,"Biography,Drama"
237278,tt10655524,movie,It Ends with Us,2024.0,130.0,"Drama,Romance"


In [14]:
def get_movie_info(movie_name): 
    movie_info = {}

    headers = {
        "User-Agent": "Mozilla/5.0" #So that Wikipedia doesn't block it
    }

    
    movie_url_name = movie_name.replace(" ", "_")
    url = "https://en.wikipedia.org/wiki/" + movie_url_name

    response = requests.get(url, headers=headers)

    print("URL:", url)
    print("STATUS:", response.status_code)

    if response.status_code != 200: #Reading valid HTML
        return None

    soup = BeautifulSoup(response.content, "html.parser")

   
    infobox = soup.find("table", class_="infobox")

    if infobox:
        wanted_fields = ["Country", "Language", "Budget", "Box office"] #The requested data

        for row in infobox.find_all("tr"):
            header = row.find("th")
            value = row.find("td")

            if header and value:
                header_text = header.get_text(" ", strip=True)
                value_text = value.get_text(" ", strip=True)

                if header_text in wanted_fields:
                    movie_info[header_text] = value_text

  
    plot_text = ""

    plot_header = soup.find("h2", id="Plot") #Summary of the film

    if plot_header:
        start_block = plot_header.parent

        for sibling in start_block.find_next_siblings():

            if sibling.name == "div" and "mw-heading" in sibling.get("class", []):
                break

            if sibling.name == "p":
                text = sibling.get_text(" ", strip=True)
                if text:
                    plot_text += text + " "

    movie_info["Plot"] = plot_text.strip()

    return movie_info

## Connecting a basics table with the requested data from other tables ##

In [15]:
df_title_basics = df_basics_filtered.loc[:, ['tconst', 'primaryTitle', 'startYear', 'genres', 'runtimeMinutes']]

In [16]:
df_title_basics.to_csv('filtered_movies_base.csv', index=False, encoding='utf-8')

### Importing Rating table ###

In [17]:
url_ratings = "https://datasets.imdbws.com/title.ratings.tsv.gz"
df_ratings = pd.read_csv(url_ratings, sep='\t', compression='gzip',na_values='\\N', low_memory=False)
df_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2213
1,tt0000002,5.5,318
2,tt0000003,6.4,2330
3,tt0000004,5.1,199
4,tt0000005,6.2,3048


In [18]:
df_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1667889 entries, 0 to 1667888
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   tconst         1667889 non-null  object 
 1   averageRating  1667889 non-null  float64
 2   numVotes       1667889 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 38.2+ MB


In [19]:
df_ratings_filtered = df_ratings[df_ratings['tconst'].isin(df_title_basics['tconst'])] #Connection by tconst
df_ratings_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12584 entries, 1531 to 1667816
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   tconst         12584 non-null  object 
 1   averageRating  12584 non-null  float64
 2   numVotes       12584 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 393.2+ KB


In [22]:
import gc


### Importing a table title principals ###

In [20]:
required_cols = ['tconst', 'nconst', 'category']

In [23]:
valid_tconsts = set(df_basics_filtered['tconst'])

chunks_list = []
url_principals = "https://datasets.imdbws.com/title.principals.tsv.gz"

reader = pd.read_csv(url_principals, sep='\t', compression='gzip', 
                     usecols=['tconst', 'nconst', 'category'],
                     na_values='\\N', chunksize=500000)  

for chunk in reader:
    filtered_chunk = chunk[chunk['tconst'].isin(valid_tconsts)]
    
    filtered_chunk = filtered_chunk[filtered_chunk['category'].isin(['actor', 'actress'])]
    
    chunks_list.append(filtered_chunk)

df_principals_filtered = pd.concat(chunks_list)

del chunks_list
gc.collect()

print(f"Done! Found {len(df_principals_filtered)} relevant rows.")

Done! Found 125491 relevant rows.


In [24]:
df_principals_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 125491 entries, 19482 to 99372588
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   tconst    125491 non-null  object
 1   nconst    125491 non-null  object
 2   category  125491 non-null  object
dtypes: object(3)
memory usage: 3.8+ MB


In [25]:
df_principals_filtered.head()

,tconst,nconst,category
19482,tt0003014,nm0096737,actress
19483,tt0003014,nm0344981,actor
19484,tt0003014,nm0511806,actor
19485,tt0003014,nm0511806,actor
19486,tt0003014,nm0489296,actor


In [26]:
df_lead_actors = df_principals_filtered.groupby('tconst')['nconst'].apply(lambda x: list(x)[:5]).reset_index()
df_lead_actors.columns = ['tconst', 'lead_actors_ids']

In [27]:
df_lead_actors.head()

,tconst,lead_actors_ids
0,tt0003014,"[nm0096737, nm0344981, nm0511806, nm0511806, n..."
1,tt0004022,"[nm0637019, nm0144663, nm1550622, nm0526921, n..."
2,tt0004150,"[nm6500432, nm6500433, nm6500433, nm6500434, n..."
3,tt0004181,"[nm0842239, nm0910400, nm0550615, nm0366008, n..."
4,tt0005541,"[nm0711615, nm0871311, nm0352445, nm0249237, n..."


In [28]:
df_final = pd.merge(df_title_basics, df_ratings_filtered, on='tconst', how='left')
df_final = pd.merge(df_final, df_lead_actors, on='tconst', how='left')
df_final.sample(20)

,tconst,primaryTitle,startYear,genres,runtimeMinutes,averageRating,numVotes,lead_actors_ids
14287,tt36595109,Il faut qu'il règne,2023.0,Documentary,65.0,NaN,NaN,[nm3882264]
13814,tt3354920,Jack and Cocaine,2014.0,Drama,94.0,7.3,13.0,"[nm4426369, nm2656219, nm4580975, nm0285890, n..."
1668,tt0070253,Jugnu,1973.0,"Action,Drama,Thriller",149.0,6.7,262.0,"[nm0004429, nm0004564, nm0695199, nm0159159, n..."
17313,tt8522432,Is College Worth It? Tom Woods and Stefan Moly...,2018.0,"News,Talk-Show",62.0,NaN,NaN,NaN
17517,tt8939168,Jenius: Countdown in the Riddle,2018.0,"Crime,Mystery,Sci-Fi",75.0,NaN,NaN,"[nm6916133, nm6916133, nm9045921, nm9045921, n..."
2477,tt0104470,I Was on Mars,1991.0,Comedy,87.0,6.2,172.0,"[nm0775056, nm0506374, nm0315964, nm0712744, n..."
12101,tt26470634,I Tried to Do Right,2023.0,Crime,64.0,NaN,NaN,"[nm14507639, nm14507643, nm14510221, nm1450763..."
17319,tt8530934,I am golden Karen,2018.0,Documentary,83.0,7.4,6.0,NaN
13214,tt31114780,Jacket Thieves,2024.0,Drama,74.0,8.6,10.0,"[nm7708413, nm5421138]"
12945,tt2979302,Journey to the Safest Place on Earth,2013.0,Documentary,100.0,6.5,75.0,NaN


In [29]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18055 entries, 0 to 18054
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tconst           18055 non-null  object 
 1   primaryTitle     18055 non-null  object 
 2   startYear        18055 non-null  float64
 3   genres           16723 non-null  object 
 4   runtimeMinutes   18055 non-null  object 
 5   averageRating    12584 non-null  float64
 6   numVotes         12584 non-null  float64
 7   lead_actors_ids  14957 non-null  object 
dtypes: float64(3), object(5)
memory usage: 1.1+ MB


## from wikipedia ##

In [32]:

import time

def build_wiki_candidates(title, year=None):
    """ייצור רשימת כתובות URL אפשריות לפי סדר עדיפות הנדסי"""
    clean_title = str(title).strip()
    # יצירת וריאציות של שם (מקור מול אותיות גדולות בתחילת מילה)
    variants = [clean_title, clean_title.title()]
    
    candidates = []
    for var in variants:
        title_url = var.replace(" ", "_")
        
        # 1. הכי ספציפי: שם + שנה + סיומת סרט
        if pd.notna(year):
            candidates.append(f"{title_url}_({int(year)}_film)")
        
        # 2. שם + סיומת סרט (מונע נפילה על דפי פירושונים כלליים)
        candidates.append(f"{title_url}_(film)")
        
        # 3. שם + שנה
        if pd.notna(year):
            candidates.append(f"{title_url}_({int(year)})")
            
        # 4. השם הנקי (עדיפות אחרונה)
        candidates.append(title_url)
    
    # הסרת כפילויות תוך שמירה על הסדר
    return list(dict.fromkeys(candidates))

def get_movie_info(movie_name, year=None):
    """שליפת נתוני סרט מויקיפדיה עם אימות תוכן מקסימלי"""
    headers = {"User-Agent": "Mozilla/5.0"}
    movie_info = {
        "Country": None, "Language": None, "Budget": None, 
        "Box office": None, "Plot": None
    }
    
    candidates = build_wiki_candidates(movie_name, year)
    soup = None
    
    # שלב החיפוש והאימות
    for candidate in candidates:
        url = "https://en.wikipedia.org/wiki/" + candidate
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                temp_soup = BeautifulSoup(response.content, "html.parser")
                
                # תנאי קריטי: מוודאים שיש טבלת מידע (Infobox) בדף
                if temp_soup.find("table", class_="infobox"):
                    soup = temp_soup
                    break 
        except Exception:
            continue
            
    if soup is None:
        return movie_info

    # מיפוי שדות מורחב לניקוי נתונים פיננסיים
    field_mapping = {
        "Country": "Country", "Countries": "Country",
        "Language": "Language", "Languages": "Language",
        "Budget": "Budget", "Production budget": "Budget",
        "Box office": "Box office", "Gross": "Box office", "Revenue": "Box office"
    }
    
    infobox = soup.find("table", class_="infobox")
    if infobox:
        for row in infobox.find_all("tr"):
            header = row.find("th")
            value = row.find("td")
            if header and value:
                header_text = header.get_text(" ", strip=True)
                if header_text in field_mapping:
                    key = field_mapping[header_text]
                    movie_info[key] = value.get_text(" ", strip=True)

    # שליפת העלילה (Plot) עם מזהים מורחבים
    plot_text = ""
    # הוספנו Summary ו-Description לכיסוי מקסימלי
    possible_plot_ids = ["Plot", "Synopsis", "Premise", "Summary", "Description"]
    
    for pid in possible_plot_ids:
        plot_header = soup.find("h2", id=pid)
        if plot_header:
            # לוקחים את האלמנט האב (h2) ורצים על האחים שלו
            start_node = plot_header.parent
            for sibling in start_node.find_next_siblings():
                # עוצרים אם הגענו לכותרת h2 חדשה או לסוף תוכן הערך
                if sibling.name == "h2" or (sibling.name == "div" and "mw-heading" in sibling.get("class", [])):
                    break
                if sibling.name == "p":
                    plot_text += sibling.get_text(" ", strip=True) + " "
            break
            
    movie_info["Plot"] = plot_text.strip() if plot_text else None
    return movie_info

In [33]:
# הגדרות לריצה החדשה
batch_size = 50
limit = 50 

all_wiki_results = []  # רשימה חדשה לתוצאות של הסבב השני

for start in range(0, min(len(df_basics_filtered), limit), batch_size):
    end = start + batch_size
    
    # לקיחת ה-Batch הרלוונטי
    batch = df_basics_filtered.iloc[start:end].copy()
    
    print(f"Processing rows {start} to {end} ")
    
    for index, row in batch.iterrows():
        movie_name = row["primaryTitle"]
        year = row["startYear"]
        tconst = row["tconst"]
        
        # שימוש בפונקציה החדשה והמשופרת
        info = get_movie_info(movie_name, year)
        
        # הוספת מזהים כדי שנוכל למזג את זה מאוחר יותר לטבלה הראשית
        info["tconst"] = tconst
        info["primaryTitle"] = movie_name
        info["startYear"] = year
        
        all_wiki_results.append(info)
        
        # השהיה קצרה כדי למנוע חסימה מוויקיפדיה (במיוחד בגלל הניסיונות המרובים לכל סרט)
        time.sleep(1) 
    
    # שמירה לגיבוי נפרד - wiki_backup_las.csv
    df_temp = pd.DataFrame(all_wiki_results)
    df_temp.to_csv("wiki_backup_last.csv", index=False)

print("Finished processing")

Processing rows 0 to 50 
Finished processing


In [34]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country       41 non-null     object 
 1   Language      41 non-null     object 
 2   Budget        4 non-null      object 
 3   Box office    4 non-null      object 
 4   Plot          24 non-null     object 
 5   tconst        50 non-null     object 
 6   primaryTitle  50 non-null     object 
 7   startYear     50 non-null     float64
dtypes: float64(1), object(7)
memory usage: 3.2+ KB


In [37]:
df_temp.head(40)

,Country,Language,Budget,Box office,Plot,tconst,primaryTitle,startYear
0,Sweden,Silent film,None,None,Sven Holm and his wife Ingeborg are happily ma...,tt0003014,Ingeborg Holm,1913.0
1,Italy,Silent Italian intertitles,None,None,None,tt0004022,Julius Caesar,1914.0
2,United States Canada,silent film English intertitles,None,None,The following plot synopsis was published in c...,tt0004150,In the Land of the Head Hunters,1914.0
3,United States,Silent (English intertitles ),None,None,The film is based on the deuterocanonical Book...,tt0004181,Judith of Bethulia,1914.0
4,United States,Silent (English intertitles ),None,None,"As described in a film magazine review, [ 5 ] ...",tt0005541,In the Palace of the King,1915.0
5,None,None,None,None,None,tt0005575,Judge Not; or the Woman of Mona Diggings,1915.0
6,United States,Silent (English intertitles ),None,None,Characters go to the movies to watch Life's Mi...,tt0006846,Idle Wives,1916.0
7,United States,Silent (English intertitles ),"$385,907 [ 1 ]",$1.75 million ( theatrical rental ),None,tt0006864,Intolerance,1916.0
8,France,Silent film with French intertitles,None,None,The story is complex and is told in 12 chapter...,tt0006886,Judex,1916.0
9,United States,Silent (English intertitles ),None,None,None,tt0008144,Jack and the Beanstalk,1917.0


In [39]:
# הגדרות לריצה החדשה
batch_size = 50
limit = 5000

all_wiki_results = []  # רשימה חדשה לתוצאות של הסבב השני

for start in range(0, min(len(df_basics_filtered), limit), batch_size):
    end = start + batch_size
    
    # לקיחת ה-Batch הרלוונטי
    batch = df_basics_filtered.iloc[start:end].copy()
    
    print(f"Processing rows {start} to {end} ")
    
    for index, row in batch.iterrows():
        movie_name = row["primaryTitle"]
        year = row["startYear"]
        tconst = row["tconst"]
        
        # שימוש בפונקציה החדשה והמשופרת
        info = get_movie_info(movie_name, year)
        
        # הוספת מזהים כדי שנוכל למזג את זה מאוחר יותר לטבלה הראשית
        info["tconst"] = tconst
        info["primaryTitle"] = movie_name
        info["startYear"] = year
        
        all_wiki_results.append(info)
        
        # השהיה קצרה כדי למנוע חסימה מוויקיפדיה (במיוחד בגלל הניסיונות המרובים לכל סרט)
        time.sleep(1) 
    
    # שמירה לגיבוי נפרד - wiki_backup_las.csv
    df_temp = pd.DataFrame(all_wiki_results)
    df_temp.to_csv("wiki_backup_last.csv", index=False)

print("Finished processing")

Processing rows 0 to 50 
Processing rows 50 to 100 
Processing rows 100 to 150 
Processing rows 150 to 200 
Processing rows 200 to 250 
Processing rows 250 to 300 
Processing rows 300 to 350 
Processing rows 350 to 400 
Processing rows 400 to 450 
Processing rows 450 to 500 
Processing rows 500 to 550 
Processing rows 550 to 600 
Processing rows 600 to 650 
Processing rows 650 to 700 
Processing rows 700 to 750 
Processing rows 750 to 800 
Processing rows 800 to 850 
Processing rows 850 to 900 
Processing rows 900 to 950 
Processing rows 950 to 1000 
Processing rows 1000 to 1050 
Processing rows 1050 to 1100 
Processing rows 1100 to 1150 
Processing rows 1150 to 1200 
Processing rows 1200 to 1250 
Processing rows 1250 to 1300 
Processing rows 1300 to 1350 
Processing rows 1350 to 1400 
Processing rows 1400 to 1450 
Processing rows 1450 to 1500 
Processing rows 1500 to 1550 
Processing rows 1550 to 1600 
Processing rows 1600 to 1650 
Processing rows 1650 to 1700 
Processing rows 1700 to

## Merging the tables ##

In [42]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country       2439 non-null   object 
 1   Language      2487 non-null   object 
 2   Budget        508 non-null    object 
 3   Box office    569 non-null    object 
 4   Plot          1786 non-null   object 
 5   tconst        5000 non-null   object 
 6   primaryTitle  5000 non-null   object 
 7   startYear     5000 non-null   float64
dtypes: float64(1), object(7)
memory usage: 312.6+ KB


In [43]:
df_final = df_final.merge(
    df_temp,
    on="tconst",
    how="left"
)

In [44]:
df_final.head()

,tconst,primaryTitle_x,startYear_x,genres,runtimeMinutes,averageRating,numVotes,lead_actors_ids,Country,Language,Budget,Box office,Plot,primaryTitle_y,startYear_y
0,tt0003014,Ingeborg Holm,1913.0,Drama,96.0,7.0,1605.0,"[nm0096737, nm0344981, nm0511806, nm0511806, n...",Sweden,Silent film,None,None,Sven Holm and his wife Ingeborg are happily ma...,Ingeborg Holm,1913.0
1,tt0004022,Julius Caesar,1914.0,"Drama,History",112.0,6.3,79.0,"[nm0637019, nm0144663, nm1550622, nm0526921, n...",Italy,Silent Italian intertitles,None,None,None,Julius Caesar,1914.0
2,tt0004150,In the Land of the Head Hunters,1914.0,"Drama,History",65.0,5.8,597.0,"[nm6500432, nm6500433, nm6500433, nm6500434, n...",United States Canada,silent film English intertitles,None,None,The following plot synopsis was published in c...,In the Land of the Head Hunters,1914.0
3,tt0004181,Judith of Bethulia,1914.0,Drama,61.0,6.2,1545.0,"[nm0842239, nm0910400, nm0550615, nm0366008, n...",United States,Silent (English intertitles ),None,None,The film is based on the deuterocanonical Book...,Judith of Bethulia,1914.0
4,tt0005541,In the Palace of the King,1915.0,"Drama,History",60.0,NaN,NaN,"[nm0711615, nm0871311, nm0352445, nm0249237, n...",United States,Silent (English intertitles ),None,None,"As described in a film magazine review, [ 5 ] ...",In the Palace of the King,1915.0


In [45]:
df_final = df_final[['tconst', 'primaryTitle_x', 'startYear_x', 'genres', 'lead_actors_ids', 
                     'runtimeMinutes', 'averageRating', 'Language', 'Country', 
                     'numVotes', 'Budget', 'Box office', 'Plot']].rename(columns={
                         'primaryTitle_x': 'primaryTitle',
                         'startYear_x': 'startYear'
                     })

## making budget into dollars ##

In [46]:
import re
import numpy as np

def convert_to_usd_millions(value):
    # 1. טיפול בערכים חסרים או לא רלוונטיים
    if pd.isna(value) or value is None:
        return np.nan
    
    val_str = str(value).lower().strip()
    
    # הסרת "רעש" נפוץ כמו סוגריים מרובעים של מקורות (למשל [ 1 ]) או מילים לא רלוונטיות
    val_str = re.sub(r'\[.*?\]', '', val_str)
    val_str = val_str.replace(',', '')
    
    # זיהוי ערכים שהם כמות כניסות/כרטיסים ולא כסף - נחזיר NaN
    if any(word in val_str for word in ['admissions', 'tickets', 'unknown']):
        return np.nan

    # 2. הגדרת שערי חליפין (נכון למאי 2026)
    # עבור מטבעות היסטוריים (כמו מרק גרמני או פרנק), נשתמש בשער האחרון הידוע או קירוב
    rates = {
        '$': 1.0, 'us$': 1.0, 'usd': 1.0,
        '€': 1.08, 'eur': 1.08,
        '£': 1.26, 'gbp': 1.26,
        '₹': 0.012, 'rs': 0.012, 'nrs': 0.012, 'rupee': 0.012, # 1 Rupee approx 0.012 USD
        '₣': 0.17, 'ff': 0.17, # French Franc approx
        'ddm': 0.5, 'east german mark': 0.5, # Historic estimate
        'reichsmarks': 0.4,
        'kčs': 0.04, 'p': 0.018, # Philippine Peso or similar
        '¥': 0.0065 # Yen
    }

    # 3. זיהוי המטבע
    current_rate = 1.0 # ברירת מחדל דולר
    for symbol, rate in rates.items():
        if symbol in val_str:
            current_rate = rate
            break

    # 4. חילוץ המספר (תומך בטווחים כמו 8-9, ניקח את הממוצע)
    numbers = re.findall(r'\d+\.?\d*', val_str)
    if not numbers:
        return np.nan
    
    if '-' in val_str and len(numbers) >= 2:
        # אם יש טווח (8-9), ניקח ממוצע
        base_number = (float(numbers[0]) + float(numbers[1])) / 2
    else:
        base_number = float(numbers[0])

    # 5. טיפול במכפילים (Units)
    multiplier = 1.0
    if 'billion' in val_str:
        multiplier = 1000.0  # מיליארד דולר = 1000 מיליון
    elif 'million' in val_str:
        multiplier = 1.0
    elif 'crore' in val_str:
        multiplier = 10.0    # 1 Crore = 10 Million
    elif 'lakh' in val_str:
        multiplier = 0.1     # 1 Lakh = 0.1 Million
    elif base_number > 5000: 
        # אם המספר גדול מאוד ואין מילה (כמו 525,000), נניח שזה ערך מלא ונחלק למיליון
        multiplier = 1/1000000

    # 6. חישוב סופי (מיליוני דולרים)
    result = base_number * multiplier * current_rate
    return round(result, 3)

# הרצה על ה-DataFrame שלך
# נשתמש בשמות העמודות המדויקים מהמחברת שלך
df_final['Budget_M$'] = df_final['Budget'].apply(convert_to_usd_millions)
df_final['BoxOffice_M$'] = df_final['Box office'].apply(convert_to_usd_millions)

# בדיקת התוצאות
print("דוגמה להמרות שבוצעו:")
display_cols = ['Budget', 'Budget_M$', 'Box office', 'BoxOffice_M$']
print(df_final[df_final['Budget_M$'].notna() | df_final['BoxOffice_M$'].notna()][display_cols].head(15))

דוגמה להמרות שבוצעו:
                   Budget  Budget_M$  \
7          $385,907 [ 1 ]      0.386   
10   $302,976 [ 1 ] [ 2 ]      0.303   
19             FF 525,000      0.089   
31      $25 million [ 2 ]     25.000   
43                   None        NaN   
57                200,000      0.200   
70         $260,000 [ 1 ]      0.260   
72     $1.5 million [ 1 ]      1.500   
105                  None        NaN   
107         $46,000 [ 1 ]      0.046   
108                  None        NaN   
121                  None        NaN   
140        $281,000 [ 1 ]      0.281   
144            $1,100,000      1.100   
162        $228,000 [ 2 ]      0.228   

                                            Box office  BoxOffice_M$  
7                  $1.75 million ( theatrical rental )         1.750  
10                                      $605,731 [ 1 ]         0.606  
19                                                None           NaN  
31                                  $4.2 million [ 2 ]

In [49]:
df_final.to_csv('final_dataset.csv', index=False, encoding='utf-8')

In [47]:
missing_count = df_final.isnull().sum()

missing_percentage = (df_final.isnull().mean() * 100).round(2)

missing_report = pd.DataFrame({
    'Column Name': missing_count.index,
    'Missing Values': missing_count.values,
    'Percentage (%)': missing_percentage.values
})


print("סיכום ערכים חסרים ב-Dataset:")
display(missing_report)

סיכום ערכים חסרים ב-Dataset:


,Column Name,Missing Values,Percentage (%)
0,tconst,0,0.00
1,primaryTitle,0,0.00
2,startYear,0,0.00
3,genres,1332,7.38
4,lead_actors_ids,3098,17.16
5,runtimeMinutes,0,0.00
6,averageRating,5471,30.30
7,Language,15568,86.23
8,Country,15616,86.49
9,numVotes,5471,30.30


In [62]:


def build_wiki_candidates2(title, year=None):
    clean_title = str(title).strip()
    # יצירת שתי גרסאות: אחת כפי שהיא ואחת עם אותיות גדולות (Title Case)
    variants = [clean_title, clean_title.title()]
    
    candidates = []
    for var in variants:
        title_url = var.replace(" ", "_")
        
        # 1. עדיפות עליונה: שם + שנה + פילם (הכי ספציפי)
        if pd.notna(year):
            candidates.append(f"{title_url}_({int(year)}_film)")
        
        # 2. עדיפות שנייה: שם + פילם (מונע נפילה על דפי פירושונים)
        candidates.append(f"{title_url}_(film)")
        
        # 3. עדיפות שלישית: שם + שנה
        if pd.notna(year):
            candidates.append(f"{title_url}_({int(year)})")
            
        # 4. עדיפות אחרונה: השם הנקי
        candidates.append(title_url)
    
    # הסרת כפילויות ושמירה על הסדר
    return list(dict.fromkeys(candidates))

def get_movie_info2(movie_name, year=None):
    headers = {"User-Agent": "Mozilla/5.0"}
    movie_info2 = {
        "Country2": None, "Language2": None, "Budget2": None, 
        "Box office2": None, "Plot2": None
    }
    
    candidates = build_wiki_candidates2(movie_name, year)
    soup = None
    
    for candidate in candidates:
        url = "https://en.wikipedia.org/wiki/" + candidate
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                temp_soup = BeautifulSoup(response.content, "html.parser")
                
                # --- ה-CONDITION הקריטי ---
                # בודקים אם יש בדף טבלת Infobox. אם אין, זה כנראה לא הדף הנכון
                if temp_soup.find("table", class_="infobox"):
                    soup = temp_soup
                    break # מצאנו את הדף הנכון, עוצרים את הלולאה
        except Exception as e:
            continue
            
    if soup is None:
        return movie_info2

    # מיפוי שדות (Field Mapping) מורחב כדי למנוע NaNs מיותרים
    field_mapping = {
        "Country": "Country2", "Countries": "Country2",
        "Language": "Language2", "Languages": "Language2",
        "Budget": "Budget2", "Production budget": "Budget2",
        "Box office": "Box office2", "Gross": "Box office2", "Revenue": "Box office2"
    }
    
    infobox = soup.find("table", class_="infobox")
    if infobox:
        for row in infobox.find_all("tr"):
            header = row.find("th")
            value = row.find("td")
            if header and value:
                header_text = header.get_text(" ", strip=True)
                if header_text in field_mapping:
                    key = field_mapping[header_text]
                    movie_info2[key] = value.get_text(" ", strip=True)

    # שליפת העלילה (Plot)
    plot_text = ""
    for pid in ["Plot", "Synopsis", "Premise"]:
        plot_header = soup.find("h2", id=pid)
        if plot_header:
            start_block = plot_header.parent
            for sibling in start_block.find_next_siblings():
                if sibling.name == "h2" or (sibling.name == "div" and "mw-heading" in sibling.get("class", [])):
                    break
                if sibling.name == "p":
                    plot_text += sibling.get_text(" ", strip=True) + " "
            break
            
    movie_info2["Plot2"] = plot_text.strip() if plot_text else None
    return movie_info2

In [63]:
def get_movie_info2(movie_name, year=None):
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # אתחול מילון עם שמות עמודות חדשים
    movie_info2 = {
        "Country2": None,
        "Language2": None,
        "Budget2": None,
        "Box office2": None,
        "Plot2": None,
    }
    
    # שימוש בפונקציית המועמדים החדשה (זו שמשתמשת ב-title() ובסדר עדיפויות נכון)
    candidates = build_wiki_candidates2(movie_name, year)
    
    soup = None
    for candidate in candidates:
        url = "https://en.wikipedia.org/wiki/" + candidate
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                temp_soup = BeautifulSoup(response.content, "html.parser")
                # CONDITION: האם יש בכלל טבלה?
                if temp_soup.find("table", class_="infobox"):
                    soup = temp_soup
                    break 
        except:
            continue
    
    if soup is None:
        return movie_info2
    
    infobox = soup.find("table", class_="infobox")
    
    # מיפוי השדות (נשאר עם מפתחות המקור של ויקיפדיה, אבל שומר לתוך עמודות 2)
    field_mapping2 = {
        "Country": "Country2", "Countries": "Country2",
        "Language": "Language2", "Languages": "Language2",
        "Budget": "Budget2", "Production budget": "Budget2",
        "Box office": "Box office2", "Gross": "Box office2", "Revenue": "Box office2"
    }
    
    if infobox:
        for row in infobox.find_all("tr"):
            header = row.find("th")
            value = row.find("td")
            if header and value:
                header_text = header.get_text(" ", strip=True)
                if header_text in field_mapping2:
                    key = field_mapping2[header_text]
                    movie_info2[key] = value.get_text(" ", strip=True)
    
    # עלילה (Plot)
    plot_text = ""
    for pid in ["Plot", "Synopsis", "Premise"]:
        plot_header = soup.find("h2", id=pid)
        if plot_header:
            # לוקחים את הבלוק שמתחת לכותרת
            for sibling in plot_header.parent.find_next_siblings():
                if sibling.name == "h2" or (sibling.name == "div" and "mw-heading" in sibling.get("class", [])):
                    break
                if sibling.name == "p":
                    plot_text += sibling.get_text(" ", strip=True) + " "
            break
            
    movie_info2["Plot2"] = plot_text.strip() if plot_text else None
    return movie_info2

In [65]:
# הגדרות לריצה החדשה
batch_size = 50
limit = 1000  # ציינת שאת רוצה להריץ על 1000 סרטים

all_wiki_results2 = []  # רשימה חדשה לתוצאות של הסבב השני

for start in range(0, min(len(df_basics_filtered), limit), batch_size):
    end = start + batch_size
    
    # לקיחת ה-Batch הרלוונטי
    batch = df_basics_filtered.iloc[start:end].copy()
    
    print(f"Processing rows {start} to {end} with New Logic (v2)...")
    
    for index, row in batch.iterrows():
        movie_name = row["primaryTitle"]
        year = row["startYear"]
        tconst = row["tconst"]
        
        # שימוש בפונקציה החדשה והמשופרת
        info2 = get_movie_info2(movie_name, year)
        
        # הוספת מזהים כדי שנוכל למזג את זה מאוחר יותר לטבלה הראשית
        info2["tconst"] = tconst
        info2["primaryTitle"] = movie_name
        info2["startYear"] = year
        
        all_wiki_results2.append(info2)
        
        # השהיה קצרה כדי למנוע חסימה מוויקיפדיה (במיוחד בגלל הניסיונות המרובים לכל סרט)
        time.sleep(1) 
    
    # שמירה לגיבוי נפרד - wiki_backup2.csv
    df_temp2 = pd.DataFrame(all_wiki_results2)
    df_temp2.to_csv("wiki_backup2.csv", index=False)

print("Finished processing with new logic!")

Processing rows 0 to 50 with New Logic (v2)...
Processing rows 50 to 100 with New Logic (v2)...
Processing rows 100 to 150 with New Logic (v2)...
Processing rows 150 to 200 with New Logic (v2)...
Processing rows 200 to 250 with New Logic (v2)...
Processing rows 250 to 300 with New Logic (v2)...
Processing rows 300 to 350 with New Logic (v2)...
Processing rows 350 to 400 with New Logic (v2)...
Processing rows 400 to 450 with New Logic (v2)...
Processing rows 450 to 500 with New Logic (v2)...
Processing rows 500 to 550 with New Logic (v2)...
Processing rows 550 to 600 with New Logic (v2)...
Processing rows 600 to 650 with New Logic (v2)...
Processing rows 650 to 700 with New Logic (v2)...
Processing rows 700 to 750 with New Logic (v2)...
Processing rows 750 to 800 with New Logic (v2)...
Processing rows 800 to 850 with New Logic (v2)...
Processing rows 850 to 900 with New Logic (v2)...
Processing rows 900 to 950 with New Logic (v2)...
Processing rows 950 to 1000 with New Logic (v2)...
Fin

In [66]:
df_temp2.sample(20)

,Country2,Language2,Budget2,Box office2,Plot2,tconst,primaryTitle,startYear
32,United States,Silent English intertitles,None,None,None,tt0012308,If Women Only Knew,1921.0
780,United States,English,"$800,000 [ 1 ]","$600,000 (domestic rentals estimate) [ 1 ]",Johnny Holiday (Allen Martin Jr) is a fatherle...,tt0041528,Johnny Holiday,1949.0
987,Austria West Germany,German,None,None,None,tt0048197,Ihr erstes Rendezvous,1955.0
41,United States,Silent (English intertitles ),None,None,"The plot, as described by the defendants in th...",tt0013259,I Am the Law,1922.0
202,United States,None,None,None,"A young aviator, Carlos Martin, is dumped by h...",tt0024189,It's Great to Be Alive,1933.0
1,Italy,Silent Italian intertitles,None,None,None,tt0004022,Julius Caesar,1914.0
269,United Kingdom,English,None,None,Odd-jobbing Irishman Jimmy moves to London to ...,tt0026548,Jimmy Boy,1935.0
290,United States,English,None,None,"An Italian, Enrico Scaffa, emigrates to Americ...",tt0027808,It Had to Happen,1936.0
738,None,English,None,None,"Pollack, an underworld gambler, is murdered, b...",tt0040489,Jinx Money,1948.0
868,United Kingdom,English,None,"£72,000 [ 1 ]","In 1938, Mme. Alice, chief designer of a famou...",tt0044759,It Started in Paradise,1952.0


In [67]:
missing_percentage = df_temp2.isnull().mean() * 100

print("אחוז הערכים החסרים בכל עמודה:")
print(missing_percentage)

אחוז הערכים החסרים בכל עמודה:
Country2        22.2
Language2       21.5
Budget2         87.7
Box office2     87.3
Plot2           44.7
tconst           0.0
primaryTitle     0.0
startYear        0.0
dtype: float64


## our new code found bigger percentage of the columns ##

In [69]:
df_final2 = df_final.merge(
    df_temp2,
    on="tconst",
    how="left"
)

In [70]:
df_final2.sample(20)

,tconst,primaryTitle_x,startYear_x,genres,lead_actors_ids,runtimeMinutes,averageRating,Language,Country,numVotes,...,Plot,Budget_M$,BoxOffice_M$,Country2,Language2,Budget2,Box office2,Plot2,primaryTitle_y,startYear_y
11828,tt2446718,Inventory of the Motherland,2012.0,Documentary,NaN,75.0,7.3,NaN,NaN,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12656,tt28504145,I Must Away,2023.0,Documentary,[nm15090930],75.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5970,tt0427378,Imágenes de la Argentina,1978.0,Documentary,NaN,80.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10771,tt2064824,Itsuka minagirumono,2006.0,NaN,NaN,84.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4536,tt0258579,In the Vortex of Sin,1972.0,"Crime,Drama","[nm0508695, nm1203196, nm0874597, nm1216212, n...",90.0,6.4,NaN,NaN,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1591,tt0067413,Interior Mechanism,1971.0,Drama,"[nm0537217, nm0783493, nm0715127, nm0660841, n...",115.0,NaN,None,None,NaN,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7224,tt10900526,Iblis: Esir-i Beden,2019.0,Horror,"[nm6165970, nm6741530, nm10952608, nm10952606,...",82.0,6.8,NaN,NaN,56.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18046,tt9908798,Jokôsei tsuppari banchô,1981.0,NaN,"[nm1006456, nm1003010, nm0993695]",63.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1720,tt0073140,I död mans spår,1975.0,"Drama,Western","[nm0512432, nm0435759, nm0027823, nm0542301, n...",71.0,4.6,None,None,72.0,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7748,tt1196944,I.T.A.L.Y. (I Trust and Love You),2008.0,"Comedy,Drama,Romance","[nm0535800, nm0704269, nm1387739, nm1698831, n...",94.0,5.2,NaN,NaN,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [71]:
df_final2 = df_final2[['tconst', 'primaryTitle_x', 'startYear_x', 'genres', 'lead_actors_ids', 
                     'runtimeMinutes', 'averageRating', 'Language', 'Country', 
                     'numVotes', 'Budget', 'Box office', 'Plot']].rename(columns={
                         'primaryTitle_x': 'primaryTitle',
                         'startYear_x': 'startYear'
                     })

In [72]:
df_final2.sample(20)

,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,Language,Country,numVotes,Budget,Box office,Plot
5790,tt0401503,Ibyeolui kil,1972.0,Drama,"[nm1069512, nm1068864, nm1069919, nm1070013]",90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16250,tt6296952,Incontriamoci,2017.0,Comedy,"[nm0232291, nm1524216, nm8621793, nm7835641, n...",80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5572,tt0369060,Infernal Affairs II,2003.0,"Action,Crime,Drama","[nm0155211, nm1284845, nm0938893, nm0628757, n...",119.0,7.2,NaN,NaN,21231.0,NaN,NaN,NaN
16373,tt6541904,It's the Heart That Dies Last,2017.0,Drama,"[nm0754882, nm0276946, nm0520317, nm0234715, n...",103.0,7.1,NaN,NaN,278.0,NaN,NaN,NaN
48,tt0013282,June Madness,1922.0,Comedy,"[nm0199087, nm0913341, nm0697796, nm0059499, n...",60.0,NaN,Silent (English intertitles ),United States,NaN,None,None,None
4389,tt0244004,John Holmes and the All Star Sex Queens,1980.0,Adult,[nm0450822],88.0,6.1,NaN,NaN,44.0,NaN,NaN,NaN
15638,tt5433276,In Love and War,2018.0,"Drama,Romance,War","[nm0422205, nm1142215, nm0937239, nm0511892, n...",135.0,6.3,NaN,NaN,742.0,NaN,NaN,NaN
11936,tt2536280,Jungle Love,2012.0,Drama,"[nm5379377, nm3602765, nm5379588, nm5379182, n...",85.0,5.6,NaN,NaN,31.0,NaN,NaN,NaN
11202,tt2191715,Into the Woods,2011.0,"Comedy,Musical","[nm4861913, nm0043137, nm0102195, nm0113605, n...",153.0,7.7,NaN,NaN,170.0,NaN,NaN,NaN
15941,tt5845926,It Is Not One Way,2018.0,Documentary,NaN,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
all_unique_values = pd.Series(pd.concat([df_final2['Budget'], df_final2['Box office']]).unique())


outliers = all_unique_values[
    ~all_unique_values.astype(str).str.contains(r'\$|€|£|million|billion', na=False, case=False)
].tolist()

print(f"נמצאו {len(outliers)} ערכים שאינם מכילים דולר, יורו או פאונד :")
print(outliers)

נמצאו 51 ערכים שאינם מכילים דולר, יורו או פאונד :
[None, 'FF 525,000', '200,000', 'Unknown', '2,077,000 East German Mark [ 1 ]', '2,411,600 DDM', '₣ 254,542', '4,300,000 KČs [ 1 ]', '₹ 8-9 lakh [ 3 ]', 'Rs 10 lakhs', 'P 208,120,000', '₹1 crore [ 1 ]', '₹1.25 crore [ 1 ]', '₹ 15 crore [ 1 ]', '₹ 5.60 crore [ 1 ]', '₹2.25 crore [ 1 ]', '₹6.30 crore [ 1 ]', '₹ 6.25 crore [ 1 ]', '410,000 Reichsmarks', '₹ 20 crore [ 1 ]', '₹ 16 crore [ 1 ]', '₹2 crore', nan, '60,219 tickets (France) [ 1 ]', '513,083 admissions (France) [ 1 ]', '2,178,371 admissions (France) [ 1 ]', '675,488 admissions (France) [ 1 ]', '2,8113,682 admissions (France) [ 2 ]', '₹2 crore [ 1 ]', '1,595,379 admissions (France) [ 1 ]', '1,073,011 admissions (France) [ 1 ]', '117,645 admissions (France) [ 1 ]', '1,074,600 admissions (France) [ 1 ]', '1,094,300 admissions (Italy) [ 1 ]', '17,091,070 KČs [ 2 ]', '31,624,135 [ 2 ]', '₹ 1.8 crores [ 2 ]', '28,164 admissions (France) [ 1 ]', '₹9.35 crore [ 1 ]', 'est. ₹4.53 crore [ 1 

In [74]:
import re
import numpy as np

def convert_to_usd_millions(value):
    # 1. טיפול בערכים חסרים או לא רלוונטיים
    if pd.isna(value) or value is None:
        return np.nan
    
    val_str = str(value).lower().strip()
    
    # הסרת "רעש" נפוץ כמו סוגריים מרובעים של מקורות (למשל [ 1 ]) או מילים לא רלוונטיות
    val_str = re.sub(r'\[.*?\]', '', val_str)
    val_str = val_str.replace(',', '')
    
    # זיהוי ערכים שהם כמות כניסות/כרטיסים ולא כסף - נחזיר NaN
    if any(word in val_str for word in ['admissions', 'tickets', 'unknown']):
        return np.nan

    # 2. הגדרת שערי חליפין (נכון למאי 2026)
    # עבור מטבעות היסטוריים (כמו מרק גרמני או פרנק), נשתמש בשער האחרון הידוע או קירוב
    rates = {
        '$': 1.0, 'us$': 1.0, 'usd': 1.0,
        '€': 1.08, 'eur': 1.08,
        '£': 1.26, 'gbp': 1.26,
        '₹': 0.012, 'rs': 0.012, 'nrs': 0.012, 'rupee': 0.012, # 1 Rupee approx 0.012 USD
        '₣': 0.17, 'ff': 0.17, # French Franc approx
        'ddm': 0.5, 'east german mark': 0.5, # Historic estimate
        'reichsmarks': 0.4,
        'kčs': 0.04, 'p': 0.018, # Philippine Peso or similar
        '¥': 0.0065 # Yen
    }

    # 3. זיהוי המטבע
    current_rate = 1.0 # ברירת מחדל דולר
    for symbol, rate in rates.items():
        if symbol in val_str:
            current_rate = rate
            break

    # 4. חילוץ המספר (תומך בטווחים כמו 8-9, ניקח את הממוצע)
    numbers = re.findall(r'\d+\.?\d*', val_str)
    if not numbers:
        return np.nan
    
    if '-' in val_str and len(numbers) >= 2:
        # אם יש טווח (8-9), ניקח ממוצע
        base_number = (float(numbers[0]) + float(numbers[1])) / 2
    else:
        base_number = float(numbers[0])

    # 5. טיפול במכפילים (Units)
    multiplier = 1.0
    if 'billion' in val_str:
        multiplier = 1000.0  # מיליארד דולר = 1000 מיליון
    elif 'million' in val_str:
        multiplier = 1.0
    elif 'crore' in val_str:
        multiplier = 10.0    # 1 Crore = 10 Million
    elif 'lakh' in val_str:
        multiplier = 0.1     # 1 Lakh = 0.1 Million
    elif base_number > 5000: 
        # אם המספר גדול מאוד ואין מילה (כמו 525,000), נניח שזה ערך מלא ונחלק למיליון
        multiplier = 1/1000000

    # 6. חישוב סופי (מיליוני דולרים)
    result = base_number * multiplier * current_rate
    return round(result, 3)

# הרצה על ה-DataFrame שלך
# נשתמש בשמות העמודות המדויקים מהמחברת שלך
df_final2['Budget_M$'] = df_final2['Budget'].apply(convert_to_usd_millions)
df_final2['BoxOffice_M$'] = df_final2['Box office'].apply(convert_to_usd_millions)

# בדיקת התוצאות
print("דוגמה להמרות שבוצעו:")
display_cols = ['Budget', 'Budget_M$', 'Box office', 'BoxOffice_M$']
print(df_final2[df_final2['Budget_M$'].notna() | df_final2['BoxOffice_M$'].notna()][display_cols].head(15))

דוגמה להמרות שבוצעו:
                   Budget  Budget_M$  \
7          $385,907 [ 1 ]      0.386   
10   $302,976 [ 1 ] [ 2 ]      0.303   
19             FF 525,000      0.089   
31      $25 million [ 2 ]     25.000   
43                   None        NaN   
57                200,000      0.200   
70         $260,000 [ 1 ]      0.260   
72     $1.5 million [ 1 ]      1.500   
105                  None        NaN   
107         $46,000 [ 1 ]      0.046   
108                  None        NaN   
121                  None        NaN   
140        $281,000 [ 1 ]      0.281   
162        $228,000 [ 2 ]      0.228   
174        $291,039 [ 1 ]      0.291   

                                            Box office  BoxOffice_M$  
7                  $1.75 million ( theatrical rental )         1.750  
10                                      $605,731 [ 1 ]         0.606  
19                                                None           NaN  
31                                  $4.2 million [ 2 ]

In [ ]:
converted_only = df_final2[df_final2['Budget_M$'].notna() | df_final2['BoxOffice_M$'].notna()]

print(f"נמצאו {len(converted_only)} שורות עם נתונים כספיים תקינים.")
print("דוגמאות להמרות שבוצעו (ללא ערכים חסרים):")

# הצגת 30 השורות הראשונות שהן לא NaN
print(converted_only[['Budget', 'Budget_M$', 'Box office', 'BoxOffice_M$']].head(30))

נמצאו 512 שורות עם נתונים כספיים תקינים.
דוגמאות להמרות שבוצעו (ללא ערכים חסרים):
                               Budget  Budget_M$  \
7                      $385,907 [ 1 ]      0.386   
10               $302,976 [ 1 ] [ 2 ]      0.303   
19                         FF 525,000      0.089   
31                  $25 million [ 2 ]     25.000   
43                               None        NaN   
57                            200,000      0.200   
70                     $260,000 [ 1 ]      0.260   
72                 $1.5 million [ 1 ]      1.500   
105                              None        NaN   
107                     $46,000 [ 1 ]      0.046   
108                              None        NaN   
121                              None        NaN   
140                    $281,000 [ 1 ]      0.281   
162                    $228,000 [ 2 ]      0.228   
174                    $291,039 [ 1 ]      0.291   
188           $200,000-$300,000 [ 1 ]      0.250   
191        $225,000 (estimated) [ 

In [77]:
df_final2.head()

,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,Language,Country,numVotes,Budget,Box office,Plot,Budget_M$,BoxOffice_M$
0,tt0003014,Ingeborg Holm,1913.0,Drama,"[nm0096737, nm0344981, nm0511806, nm0511806, n...",96.0,7.0,Silent film,Sweden,1605.0,None,None,Sven Holm and his wife Ingeborg are happily ma...,NaN,NaN
1,tt0004022,Julius Caesar,1914.0,"Drama,History","[nm0637019, nm0144663, nm1550622, nm0526921, n...",112.0,6.3,Silent Italian intertitles,Italy,79.0,None,None,None,NaN,NaN
2,tt0004150,In the Land of the Head Hunters,1914.0,"Drama,History","[nm6500432, nm6500433, nm6500433, nm6500434, n...",65.0,5.8,silent film English intertitles,United States Canada,597.0,None,None,The following plot synopsis was published in c...,NaN,NaN
3,tt0004181,Judith of Bethulia,1914.0,Drama,"[nm0842239, nm0910400, nm0550615, nm0366008, n...",61.0,6.2,Silent (English intertitles ),United States,1545.0,None,None,The film is based on the deuterocanonical Book...,NaN,NaN
4,tt0005541,In the Palace of the King,1915.0,"Drama,History","[nm0711615, nm0871311, nm0352445, nm0249237, n...",60.0,NaN,Silent (English intertitles ),United States,NaN,None,None,"As described in a film magazine review, [ 5 ] ...",NaN,NaN


In [52]:
missing_percentage = df_final.isnull().mean() * 100

print("אחוז הערכים החסרים בכל עמודה:")
print(missing_percentage)

אחוז הערכים החסרים בכל עמודה:
tconst              0.000000
primaryTitle        0.000000
startYear           0.000000
genres              7.377458
lead_actors_ids    17.158682
runtimeMinutes      0.000000
averageRating      30.312933
Language           89.266131
Country            89.598449
numVotes           30.312933
Budget             97.989477
Box office         97.740238
Plot               92.389920
dtype: float64


## now for all 5000 ##

In [76]:
batch_size = 50
limit = 5000  

all_wiki_results3 = []  

for start in range(0, min(len(df_basics_filtered), limit), batch_size):
    end = start + batch_size
    
    batch = df_basics_filtered.iloc[start:end].copy()
    
    print(f"Processing rows {start} to {end} ")
    
    for index, row in batch.iterrows():
        movie_name = row["primaryTitle"]
        year = row["startYear"]
        tconst = row["tconst"]
        
        info3 = get_movie_info2(movie_name, year) 
        
        info3["tconst"] = tconst
        info3["primaryTitle"] = movie_name
        info3["startYear"] = year
        
        all_wiki_results3.append(info3)
        
        time.sleep(1) 
    
    df_temp3 = pd.DataFrame(all_wiki_results3)
    df_temp3.to_csv("wiki_backup3.csv", index=False)

print("Finished processing with new logic!")

Processing rows 0 to 50 
Processing rows 50 to 100 
Processing rows 100 to 150 


KeyboardInterrupt: 